# hp-Convergence Study — pyCAFE vs FEniCSx

**Problem:** 2D rectangular cavity 1 m × 0.5 m, air (c₀ = 343 m/s), all hard walls.  
**Reference:** analytical eigenfrequencies f(m,n) = (c₀/2)√((m/Lx)²+(n/Ly)²).  
**Metric:** mean relative error over first 6 non-degenerate modes.

## Study plan
| Study | Variable | Fixed |
|---|---|---|
| **h-convergence** | mesh size h | polynomial order p |
| **p-convergence** | polynomial order p | mesh size h |

**Expected rates** (eigenvalue problems, smooth solutions):
- p=1 (Q1/CQUAD4): error ∝ h² → slope 2 on log-log
- p=2 (Q2/CQUAD8): error ∝ h⁴ → slope 4 on log-log
- p=3 (Q3):        error ∝ h⁶ → slope 6 on log-log

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import time, math, os, sys, tempfile

sys.path.insert(0, os.path.abspath("../.."))

Lx, Ly = 1.0, 0.5
c0 = 343.0
NUM_MODES = 6

# ── analytical reference: UNIQUE frequencies (non-degenerate) ────────────────
# Example: (0,1) and (2,0) both give 343 Hz for this geometry → kept once.
# The FEM comparison uses greedy closest-match so degenerate mode ordering
# in the eigensolver does not corrupt the error metric.
def analytical_modes(Lx, Ly, c0, n=20):
    modes = []
    for m in range(20):
        for k in range(20):
            if m == 0 and k == 0:
                continue
            f = (c0/2)*np.sqrt((m/Lx)**2 + (k/Ly)**2)
            modes.append((round(f, 4), m, k))
    modes.sort()
    seen, out = set(), []
    for f, m, k in modes:
        if f not in seen:
            seen.add(f); out.append((f, m, k))
        if len(out) == n:
            break
    return out

modes_ref = analytical_modes(Lx, Ly, c0, NUM_MODES)
f_ref = np.array([f for f, m, k in modes_ref])

print("Reference modes (unique frequencies):")
for i, (f, m, k) in enumerate(modes_ref):
    print(f"  mode {i+1}: ({m},{k}) -> {f:.4f} Hz")

# ── greedy closest-match: robust to degenerate modes and solver ordering ──────
def match_errors(freqs_fem, f_ref_arr):
    """For each analytical frequency find the closest unused FEM frequency.
    Returns mean relative error [%]. Handles degenerate modes gracefully."""
    available = list(freqs_fem)
    errors = []
    for f_an in f_ref_arr:
        if not available:
            break
        diffs = [abs(f - f_an) / f_an for f in available]
        best = int(np.argmin(diffs))
        errors.append(diffs[best])
        available.pop(best)
    if not errors:
        return 100.0
    return float(np.mean(errors)) * 100.0


In [ ]:
import gmsh

def make_gmsh_mesh(Nx, Ny, order, msh_path):
    """Generate a structured rectangular CQUAD mesh via Gmsh.
    order=1 -> CQUAD4, order=2 -> CQUAD8 (serendipity)."""
    if gmsh.isInitialized():
        gmsh.finalize()
    gmsh.initialize()
    gmsh.model.add("rect")
    gmsh.option.setNumber("General.Verbosity", 0)

    p1 = gmsh.model.geo.addPoint(0,  0,  0)
    p2 = gmsh.model.geo.addPoint(Lx, 0,  0)
    p3 = gmsh.model.geo.addPoint(Lx, Ly, 0)
    p4 = gmsh.model.geo.addPoint(0,  Ly, 0)
    l1 = gmsh.model.geo.addLine(p1, p2)
    l2 = gmsh.model.geo.addLine(p2, p3)
    l3 = gmsh.model.geo.addLine(p3, p4)
    l4 = gmsh.model.geo.addLine(p4, p1)
    cl   = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
    surf = gmsh.model.geo.addPlaneSurface([cl])
    gmsh.model.addPhysicalGroup(1, [l1], name="bottom")
    gmsh.model.addPhysicalGroup(1, [l2], name="right")
    gmsh.model.addPhysicalGroup(1, [l3], name="top")
    gmsh.model.addPhysicalGroup(1, [l4], name="left")
    gmsh.model.addPhysicalGroup(2, [surf], name="domain")
    gmsh.model.geo.synchronize()

    gmsh.model.mesh.setTransfiniteCurve(l1, Nx+1)
    gmsh.model.mesh.setTransfiniteCurve(l3, Nx+1)
    gmsh.model.mesh.setTransfiniteCurve(l2, Ny+1)
    gmsh.model.mesh.setTransfiniteCurve(l4, Ny+1)
    gmsh.model.mesh.setTransfiniteSurface(surf)
    gmsh.model.mesh.recombine()

    gmsh.option.setNumber("Mesh.ElementOrder", order)
    gmsh.option.setNumber("Mesh.RecombineAll",  1)
    if order == 2:
        gmsh.option.setNumber("Mesh.SecondOrderIncomplete", 1)  # CQUAD8

    gmsh.model.mesh.generate(2)
    gmsh.write(str(msh_path))
    gmsh.finalize()

print("Mesh generator ready.")

In [ ]:
import pycafe
from pycafe.solver.solver_modale import solve_modal_acoustic_reduced

NX_LIST = [2, 3, 4, 6, 8, 12, 16, 24, 32]

def run_pycafe(Nx, order, tmpdir):
    Ny = max(1, Nx // 2)
    msh = os.path.join(tmpdir, f"mesh_{Nx}_{order}.msh")
    make_gmsh_mesh(Nx, Ny, order, msh)

    fluid = pycafe.load_fluid("air")
    nodes, elements, boundaries = pycafe.load_mesh(msh)
    bc = ([], [], 0.0, [], 0.0+0j, [], 0.0, None, 0.0)
    system = pycafe.prepare_acoustic_system(
        nodes=nodes, elements=elements, boundaries=boundaries,
        rho=fluid["rho"], c0=fluid["c0"], bc=bc, debug=False,
    )
    n_dof = system["K_red"].shape[0]
    # request extra modes to account for trivial mode + possible degenerate pair
    n_modes_req = min(NUM_MODES * 2 + 4, n_dof - 1)
    freqs, _ = solve_modal_acoustic_reduced(
        system["K_red"], system["M_red"], num_modes=n_modes_req
    )
    err = match_errors(freqs, f_ref)
    h = Lx / Nx
    return h, n_dof, err

tmpdir = tempfile.mkdtemp()
import warnings; warnings.filterwarnings("ignore")

res_cafe4 = []   # CQUAD4 (p=1)
res_cafe8 = []   # CQUAD8 (p=2)

print(f"{'Nx':>4}  {'h':>8}  {'DOF(Q4)':>9}  {'err%(Q4)':>10}  {'DOF(Q8)':>9}  {'err%(Q8)':>10}")
print("-" * 60)

for Nx in NX_LIST:
    h4, dof4, e4 = run_pycafe(Nx, 1, tmpdir)
    h8, dof8, e8 = run_pycafe(Nx, 2, tmpdir)
    res_cafe4.append((h4, dof4, e4))
    res_cafe8.append((h8, dof8, e8))
    print(f"{Nx:>4}  {h4:>8.4f}  {dof4:>9}  {e4:>10.6f}%  {dof8:>9}  {e8:>10.6f}%")

res_cafe4 = np.array(res_cafe4)
res_cafe8 = np.array(res_cafe8)


In [ ]:
from mpi4py import MPI
from dolfinx import mesh as dmesh, fem
from dolfinx.fem import functionspace
from dolfinx.fem.petsc import assemble_matrix as petsc_assemble_matrix
import ufl
from slepc4py import SLEPc

def run_fenics(Nx, p):
    Ny = max(1, Nx // 2)
    msh = dmesh.create_rectangle(
        MPI.COMM_WORLD, [[0., 0.], [Lx, Ly]], [Nx, Ny],
        cell_type=dmesh.CellType.quadrilateral,
    )
    if p == 1:
        V = functionspace(msh, ("Lagrange", 1))
    elif p == 2:
        try:
            V = functionspace(msh, ("Serendipity", 2))
        except Exception:
            V = functionspace(msh, ("Lagrange", 2))
    else:
        V = functionspace(msh, ("Lagrange", p))

    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    K_mat = petsc_assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v))*ufl.dx))
    K_mat.assemble()
    M_mat = petsc_assemble_matrix(fem.form((1./c0**2)*ufl.inner(u,v)*ufl.dx))
    M_mat.assemble()

    n_dof = V.dofmap.index_map.size_global
    # request extra modes: account for trivial mode + degenerate pair at 343 Hz
    nev = min(NUM_MODES * 2 + 8, n_dof - 1)

    eps = SLEPc.EPS().create(MPI.COMM_WORLD)
    eps.setOperators(K_mat, M_mat)
    eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)
    eps.setWhichEigenpairs(SLEPc.EPS.Which.SMALLEST_REAL)
    eps.setDimensions(nev=nev)
    eps.setTolerances(tol=1e-12, max_it=500)
    eps.setFromOptions()
    eps.solve()

    xr, xi_v = K_mat.createVecs()
    freqs = []
    for i in range(eps.getConverged()):
        lam = float(np.real(eps.getEigenpair(i, xr, xi_v)))
        if lam > 1000.:
            freqs.append(np.sqrt(lam) / (2*np.pi))
    freqs = np.array(sorted(freqs))
    err = match_errors(freqs, f_ref)
    return Lx/Nx, n_dof, err

res_fenics = {1: [], 2: [], 3: []}

for p in [1, 2, 3]:
    print(f"\nFEniCSx Q{p}:")
    print(f"{'Nx':>4}  {'h':>8}  {'DOF':>9}  {'err%':>10}")
    print("-" * 36)
    for Nx in NX_LIST:
        h, dof, err = run_fenics(Nx, p)
        res_fenics[p].append((h, dof, err))
        print(f"{Nx:>4}  {h:>8.4f}  {dof:>9}  {err:>10.6f}%")
    res_fenics[p] = np.array(res_fenics[p])


In [ ]:
# p-convergence at fixed mesh (Nx=16, Ny=8)
NX_FIXED = 16
P_LIST = [1, 2, 3, 4]

res_p = []
print(f"p-convergence at Nx={NX_FIXED}:")
print(f"{'p':>3}  {'DOF':>9}  {'err%':>12}")
print("-" * 28)

# pyCAFE p=1,2
for order in [1, 2]:
    h, dof, err = run_pycafe(NX_FIXED, order, tmpdir)
    res_p.append((f"pyCAFE p={order}", order, dof, err))
    print(f"{order:>3}  {dof:>9}  {err:>12.8f}%  (pyCAFE CQUAD{4 if order==1 else 8})")

# FEniCSx p=1,2,3,4
for p in P_LIST:
    h, dof, err = run_fenics(NX_FIXED, p)
    res_p.append((f"FEniCSx p={p}", p, dof, err))
    print(f"{p:>3}  {dof:>9}  {err:>12.8f}%  (FEniCSx Q{p})")

In [ ]:
# Floor very small errors to avoid -inf in log scale
ERR_FLOOR = 1e-6  # 1e-6 %  (effectively machine precision)

def floor_err(arr):
    a = arr.copy()
    a[a < ERR_FLOOR] = ERR_FLOOR
    return a

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

def slope_line(ax, x_data, y_data, slope, label, color, offset=1.5):
    mid = len(x_data) // 2
    x0, y0 = x_data[mid], y_data[mid] * offset
    xs = np.array([x_data[0], x_data[-1]])
    ys = y0 * (xs / x0)**slope
    ax.plot(xs, ys, "--", color=color, lw=1.2, alpha=0.7, label=label)

# ── 1. h-convergence ─────────────────────────────────────────────────────────
ax = axes[0]
h4 = res_cafe4[:,0]; e4 = floor_err(res_cafe4[:,2])
h8 = res_cafe8[:,0]; e8 = floor_err(res_cafe8[:,2])
hf1 = res_fenics[1][:,0]; ef1 = floor_err(res_fenics[1][:,2])
hf2 = res_fenics[2][:,0]; ef2 = floor_err(res_fenics[2][:,2])
hf3 = res_fenics[3][:,0]; ef3 = floor_err(res_fenics[3][:,2])

ax.loglog(h4,  e4,  "b-o", ms=6, lw=1.8, label="pyCAFE CQUAD4 (p=1)")
ax.loglog(h8,  e8,  "b-s", ms=6, lw=1.8, label="pyCAFE CQUAD8 (p=2)")
ax.loglog(hf1, ef1, "r-o", ms=6, lw=1.8, label="FEniCSx Q1 (p=1)")
ax.loglog(hf2, ef2, "r-s", ms=6, lw=1.8, label="FEniCSx Q2 (p=2)")
ax.loglog(hf3, ef3, "r-^", ms=6, lw=1.8, label="FEniCSx Q3 (p=3)")

h_all = np.array([h4[0], h4[-1]])
slope_line(ax, h4,  e4,  2, "slope 2", "#888888")
slope_line(ax, h8,  e8,  4, "slope 4", "#555555")
slope_line(ax, hf3, ef3, 6, "slope 6", "#222222")

ax.set_xlabel("Element size h [m]", fontsize=11)
ax.set_ylabel("Mean relative error [%]", fontsize=11)
ax.set_title("h-convergence (error vs mesh size)", fontsize=12)
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, which="both", alpha=0.3)

# ── 2. DOF-convergence ───────────────────────────────────────────────────────
ax2 = axes[1]
ax2.loglog(res_cafe4[:,1], e4,  "b-o", ms=6, lw=1.8, label="pyCAFE CQUAD4 (p=1)")
ax2.loglog(res_cafe8[:,1], e8,  "b-s", ms=6, lw=1.8, label="pyCAFE CQUAD8 (p=2)")
ax2.loglog(res_fenics[1][:,1], ef1, "r-o", ms=6, lw=1.8, label="FEniCSx Q1 (p=1)")
ax2.loglog(res_fenics[2][:,1], ef2, "r-s", ms=6, lw=1.8, label="FEniCSx Q2 (p=2)")
ax2.loglog(res_fenics[3][:,1], ef3, "r-^", ms=6, lw=1.8, label="FEniCSx Q3 (p=3)")

ax2.set_xlabel("Number of DOF", fontsize=11)
ax2.set_ylabel("Mean relative error [%]", fontsize=11)
ax2.set_title("DOF-convergence (accuracy per DOF)", fontsize=12)
ax2.legend(fontsize=8, loc="upper right")
ax2.grid(True, which="both", alpha=0.3)

# ── 3. p-convergence ─────────────────────────────────────────────────────────
ax3 = axes[2]
p_cafe  = [r[1] for r in res_p if "pyCAFE"  in r[0]]
e_cafe  = [max(r[3], ERR_FLOOR) for r in res_p if "pyCAFE"  in r[0]]
p_fen   = [r[1] for r in res_p if "FEniCSx" in r[0]]
e_fen   = [max(r[3], ERR_FLOOR) for r in res_p if "FEniCSx" in r[0]]

ax3.semilogy(p_cafe, e_cafe, "b-o", ms=7, lw=1.8, label=f"pyCAFE  (Nx={NX_FIXED})")
ax3.semilogy(p_fen,  e_fen,  "r-s", ms=7, lw=1.8, label=f"FEniCSx (Nx={NX_FIXED})")

ax3.set_xlabel("Polynomial order p", fontsize=11)
ax3.set_ylabel("Mean relative error [%]", fontsize=11)
ax3.set_title(f"p-convergence (fixed h = {Lx/NX_FIXED:.4f} m)", fontsize=12)
ax3.set_xticks([1, 2, 3, 4])
ax3.legend(fontsize=9)
ax3.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig("convergence_hp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved -> convergence_hp.png")


In [ ]:
# Compute measured convergence rates from log-log slope
def measured_slope(h_arr, err_arr):
    # linear fit on log-log (use last half for asymptotic rate)
    n = len(h_arr)
    lh = np.log(h_arr[n//2:])
    le = np.log(err_arr[n//2:])
    valid = np.isfinite(lh) & np.isfinite(le)
    if valid.sum() < 2:
        return float('nan')
    return np.polyfit(lh[valid], le[valid], 1)[0]

print("=" * 52)
print(f"{'Method':25}  {'Expected':>8}  {'Measured':>8}")
print("-" * 52)

data = [
    ("pyCAFE CQUAD4 (p=1)", res_cafe4[:,0],  res_cafe4[:,2],  2),
    ("pyCAFE CQUAD8 (p=2)", res_cafe8[:,0],  res_cafe8[:,2],  4),
    ("FEniCSx Q1   (p=1)",  res_fenics[1][:,0], res_fenics[1][:,2], 2),
    ("FEniCSx Q2   (p=2)",  res_fenics[2][:,0], res_fenics[2][:,2], 4),
    ("FEniCSx Q3   (p=3)",  res_fenics[3][:,0], res_fenics[3][:,2], 6),
]

for name, h_arr, e_arr, expected in data:
    rate = measured_slope(h_arr, e_arr)
    print(f"{name:25}  {expected:>8}  {rate:>8.2f}")

print("=" * 52)
print("\n(rates measured by linear fit on log-log in the asymptotic regime)")